<a href="https://colab.research.google.com/github/aiportfoliorhea/ai-portfolio/blob/main/loan_bot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn chromadb langchain pypdf sentence-transformers langchain-community langchain-text-splitters langchain-chroma anthropic ragas datasets -q langchain_anthropic

In [ ]:
import chromadb
import langchain
import fastapi

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
from bs4 import BeautifulSoup

with open("jpm-10K.txt", "r", encoding="latin-1") as f:
    raw = f.read()

soup = BeautifulSoup(raw, "html.parser")
for hidden in soup.find_all("div", style=lambda s: s and "display:none" in s):
    hidden.decompose()
clean_text = soup.get_text(separator="\n", strip=True)

# Truncate the clean text, not the raw HTML
small = clean_text[:100000]

with open("jpm-10K-small-clean.txt", "w", encoding="utf-8") as f:
    f.write(small)

In [ ]:
print(clean_text[:1000])

In [ ]:
from langchain_chroma import Chroma
import chromadb
vector_store = Chroma(
    collection_name="loanbot",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

print("Vector store ready")

In [ ]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("jpm-10K-small-clean.txt", encoding="latin-1")
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=10)
chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks")

In [ ]:
# @title
for i in range(len(chunks)):
    print(f"Chunk {i}: {chunks[i]}")

In [ ]:
vector_store.add_documents(chunks)
print("Chunks stored in vector store")

In [ ]:
retriever = vector_store.as_retriever()
results = retriever.invoke("What is JPMorgan's total loan portfolio size?")
print(results)

In [ ]:
for doc in results:
    print(doc.page_content)
    print("---")

In [ ]:
import anthropic
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"]  = userdata.get('anthropic_key')
print("fetched Anthropic key succesfully")

In [ ]:
from anthropic import Anthropic

client = Anthropic()

def ask_loanbot(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": f"""Act as a legal loan document assistant. Answer the question using only the context provided.
If the answer is not in the context, say "I don't have that information in the document". Don't make up any answers, strictly stick to the context given to you."

Context:
{context}

Question: {question}"""
            }
        ]
    )

    return message.content[0].text

print(ask_loanbot("Where is JPMorgan Chase headquartered?"))
print("-----------------------------------")
print(ask_loanbot("What is the weather in New York?"))
print(" ")
print("-------------------------------")
print(ask_loanbot("What is JPMorgan's total revenue for 2025?"))


In [ ]:
test_questions = [
    # Answerable
    "Where is JPMorgan Chase headquartered?",
    "What is JPMorgan Chase's ticker symbol and on which exchange does it trade?",
    "What are JPMorgan Chase's three reportable business segments?",
    "How many employees does JPMorgan Chase have globally?",
    "What were JPMorgan Chase's total assets as of December 31, 2025?",
    # Not answerable
    "What is JPMorgan Chase's net interest income for 2025?",
    "What is JPMorgan Chase's CET1 capital ratio?",
    "Who is the CEO of JPMorgan Chase?",
    "What is JPMorgan Chase's total revenue for 2025?",
    "What is JPMorgan Chase's return on equity?",
]

ground_truths = [
    # Answerable
    "JPMorgan Chase is headquartered at 270 Park Avenue, New York, New York 10017.",
    "JPMorgan Chase trades under the ticker symbol JPM on the New York Stock Exchange.",
    "JPMorgan Chase's three reportable business segments are Consumer and Community Banking (CCB), Commercial and Investment Bank (CIB), and Asset and Wealth Management (AWM).",
    "JPMorgan Chase had 318,512 employees globally as of December 31, 2025.",
    "JPMorgan Chase had $4.4 trillion in assets as of December 31, 2025.",
    # Not answerable
    "JPMorgan Chase's net interest income for 2025 is not available in this excerpt.",
    "JPMorgan Chase's CET1 capital ratio is not available in this excerpt.",
    "Jamie Dimon is the CEO of JPMorgan Chase.",
    "JPMorgan Chase's total revenue for 2025 is not available in this excerpt.",
    "JPMorgan Chase's return on equity is not available in this excerpt.",
]

In [ ]:
# Generate answers + capture retrieved context
answers = []
contexts = []

for question in test_questions:
    retrieved_docs = retriever.invoke(question)
    context_list = [doc.page_content for doc in retrieved_docs]
    contexts.append(context_list)

    answer = ask_loanbot(question)
    answers.append(answer)
    print(f"Qestion : {question}")
    print(f"Answer : {answer}...")
    print("---------------------")

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_anthropic import ChatAnthropic
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset

# RAGAS with Claude instead of OpenAI (already paid for the key :/)
judge_llm = LangchainLLMWrapper(
    ChatAnthropic(
        model="claude-sonnet-4-6",
        anthropic_api_key=os.environ["ANTHROPIC_API_KEY"]
    )
)

judge_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

# Apply to each metric
for metric in [faithfulness, answer_relevancy, context_precision, context_recall]:
    metric.llm = judge_llm

answer_relevancy.embeddings = judge_embeddings

# Build dataset
data = {
    "question": test_questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths,
}
dataset = Dataset.from_dict(data)

results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)

print(results)

In [ ]:
import pandas as pd

df = results.to_pandas()

# Split into answerable vs not
df["answerable"] = (
    ["Yes"] * 5 + ["No"] * 5
)

print("=== FULL RESULTS ===")
print(df[["user_input", "faithfulness", "answer_relevancy",
          "context_precision", "context_recall", "answerable"]].to_string())

print("\n=== AVERAGES BY ANSWERABILITY ===")
print(df.groupby("answerable")[
    ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
].mean().round(3))

In [ ]:
print(answers[9])  # CET1 is index 6
print("---")
print(contexts[9])  # what chunks were retrieved